In [31]:
import numpy as np
import pandas as pd
import skforecast
print(skforecast.__version__)

0.20.1


In [32]:
lags_contiguous = np.arange(1, 25)          # [1..24]
lags_noncontig  = np.array([1, 2, 3, 6, 12, 24])

max_lag     = 24
window_size = max_lag   # no window features: window_size == max_lag

rng = np.random.default_rng(42)
y = rng.standard_normal(100_000)
y_strided = np.lib.stride_tricks.sliding_window_view(y, window_size)[:-1]
print(f"y_strided shape: {y_strided.shape}")

y_strided shape: (99976, 24)


In [33]:
# ── Correctness check ──────────────────────────────────────────────────────────
# Both methods must produce the same matrix for contiguous lags.
result_slice = y_strided[:, window_size - max_lag:][:, ::-1]  # view
result_fancy = y_strided[:, window_size - lags_contiguous]     # copy

assert np.array_equal(result_slice, result_fancy), "Results differ!"
print("Correctness check passed: both methods produce identical matrices.")
print(f"result_slice shares memory with y_strided : {np.shares_memory(result_slice, y_strided)}")
print(f"result_fancy shares memory with y_strided : {np.shares_memory(result_fancy, y_strided)}")

Correctness check passed: both methods produce identical matrices.
result_slice shares memory with y_strided : True
result_fancy shares memory with y_strided : False


In [34]:
%%timeit
# Contiguous lags 1-24 — slice + reverse (zero-copy view)
X_data = y_strided[:, window_size - max_lag:][:, ::-1]

747 ns ± 81.2 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


In [35]:
%%timeit
# Contiguous lags 1-24 — fancy index (forces a copy)
X_data = y_strided[:, window_size - lags_contiguous]

1.97 ms ± 231 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [36]:
%%timeit
# Non-contiguous lags [1,2,3,6,12,24] — fancy index (copy, fewer columns)
X_data = y_strided[:, window_size - lags_noncontig]

452 μs ± 6.8 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [37]:
from skforecast.recursive import ForecasterRecursive
from skforecast.direct import ForecasterDirect
from sklearn.linear_model import LinearRegression
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
y = pd.Series(rng.standard_normal(10_000), index=pd.date_range("2000-01-01", periods=10_000, freq="h"))
forecaster = ForecasterRecursive(
    estimator=LinearRegression(),
    lags=lags_contiguous,
)
forecaster_direct = ForecasterDirect(
    estimator=LinearRegression(),
    lags=lags_contiguous,
    steps=24
)

In [38]:
%%timeit

X_train, y_train = forecaster._create_lags(y)

246 μs ± 7.36 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [39]:
%%timeit

_ = forecaster_direct._create_lags(y)

192 μs ± 5.3 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


Impact in the prediction

In [40]:
forecaster = ForecasterRecursive(
    estimator=LinearRegression(),
    lags=lags_contiguous,
)


In [41]:
%%timeit
forecaster.fit(y=y, store_in_sample_residuals=True)

23.2 ms ± 5.68 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [42]:
%%timeit

_ = forecaster.predict(steps=500)

3.46 ms ± 105 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [43]:
%%timeit
_ = forecaster.predict_bootstrapping(steps=24, n_boot=500)

4.96 ms ± 175 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
